<p class="lead">A model is a claim about data it has never seen, and validation is the experiment that tests the claim. Almost every inflated result in the literature of machine learning for chemistry and medicine comes from an experiment whose design did not match the claim: a random split used to support a statement about new chemotypes, a hyperparameter chosen on the same data used to report the score, a metric that flatters an imbalanced problem, or two models declared different on the strength of a paired test whose assumptions do not hold. This unit builds the validation apparatus of the laboratory: the taxonomy of splits and what each one simulates, cross-validation with model selection separated from model assessment, every metric you will be asked for derived from its definition, honest statistical comparison of two models, and y-randomisation. Its central experiment is the one every QSAR referee asks for — <strong>the same model under a random split and under a scaffold split</strong> — and the gap between the two numbers is the lesson.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad enseña a validar un modelo de forma honesta. Se construye la taxonomía de particiones (aleatoria, estratificada, por grupos, por andamiaje de Bemis-Murcko, por agrupamiento de Butina y una partición temporal simulada) y se usa la distancia al conjunto de entrenamiento como diagnóstico; se distingue la selección de modelos de su evaluación mediante validación cruzada anidada; se derivan las métricas de regresión y de clasificación desde su definición, incluyendo el comportamiento bajo desbalance de clases, la calibración y el puntaje de Brier; se comparan dos modelos con pruebas pareadas, intervalos bootstrap y la corrección de Nadeau-Bengio; y se realiza la aleatorización de la variable respuesta con 100 permutaciones. El experimento central compara el mismo modelo bajo partición aleatoria y bajo partición por andamiaje: el área bajo la curva precisión-exhaustividad cae de 0,835 a 0,650 y el error cuadrático medio de regresión sube de 0,731 a 0,793 unidades logarítmicas.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Choose the split that matches the deployment question — random, stratified, group, scaffold/cluster, temporal — and explain why random splits overestimate performance for molecules and for patients.</li>
<li>Implement each split in R and diagnose it with the distance from each test point to the training set.</li>
<li>Use k-fold, repeated, stratified and nested cross-validation correctly, and keep model selection separate from model assessment.</li>
<li>Derive and interpret regression metrics (RMSE, MAE, \(R^2\), Spearman) and classification metrics (confusion matrix, precision, recall, F1, MCC, ROC-AUC, PR-AUC, calibration, Brier), including under class imbalance.</li>
<li>Compare two models statistically with paired cross-validation differences, a bootstrap confidence interval and the Nadeau–Bengio corrected t-test.</li>
<li>Run y-randomisation and place the real score inside a null distribution of 100 scrambles.</li>
<li>Write the validation paragraph of a paper so that a reader can reconstruct exactly what was measured.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly, running each chunk and reading its output; then change the split and watch every number move. This is the <strong>R edition</strong> of the unit: the same data, splits and experiments as the <a href="../../Python/U12_Validation/index.html">Python version</a>, written with base R, <code>data.table</code>, <code>caret</code><sup class="c"><a href="#r33">[33]</a></sup> for the resampling machinery, <code>ranger</code><sup class="c"><a href="#r34">[34]</a></sup> for the forests, <code>glmnet</code><sup class="c"><a href="#r35">[35]</a></sup> for the penalised linear models and <code>pROC</code><sup class="c"><a href="#r30">[30]</a></sup> for the ROC curves. Two things that Python gets from libraries are written by hand here and are better for it: the Butina clustering of Section 2.4 (RDKit is not available in R, so the algorithm is implemented in eight lines of sparse algebra) and the precision–recall area of Section 5.3. In a fresh Colab R runtime run once: <code>install.packages(c("data.table","caret","ranger","glmnet","pROC","patchwork","R.utils"))</code>. <a href="../U11_Supervised_Workflow/index.html">Unit 11</a> built the preprocessing chain whose single honest number this unit now interrogates; Units 13 and 14 use the splits defined here for every comparison they make.</p></div>



In [ ]:
## Setup: libraries, global seed, plotting defaults, data loader.
library(data.table); library(Matrix); library(ggplot2); library(patchwork)
library(caret);      library(ranger); library(glmnet);  library(pROC)
set.seed(0)
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
data_path <- function(name) file.path("..", "data", name)
load_data  <- function(name, ...) data.table::fread(data_path(name), ...)
cat("R", as.character(getRversion()), "| caret", as.character(packageVersion("caret")),
    "| ranger", as.character(packageVersion("ranger")), "| glmnet", as.character(packageVersion("glmnet")),
    "| pROC", as.character(packageVersion("pROC")), "\n")


In [ ]:
## The modelling table of Units 11-14, its two targets and its features.
df  <- load_data("rock1_chembl_curated.csv")
reg <- df[censored == FALSE]
y       <- reg$pActivity                       ## regression target
y_cls   <- as.integer(y >= 7.0)                ## classification target: potency gate at 100 nM
scaffold <- reg$scaffold_smiles                ## Bemis-Murcko generic scaffold
fpl  <- load_data("rock1_morgan_sparse.csv.gz")   ## frozen Morgan bits (see Unit 11, Section 2)
ikey <- reg$inchikey; fpl <- fpl[id %in% ikey]
X  <- sparseMatrix(i = match(fpl$id, ikey), j = fpl$bit + 1L, x = 1,
                   dims = c(length(ikey), 2048), dimnames = list(NULL, paste0("bit", 0:2047)))
Xm <- as.matrix(X)                              ## dense copy for glmnet/ranger (2764 x 2048 doubles)
cat("compounds:", length(y), "| features:", ncol(X), "| actives (pActivity >= 7):", sum(y_cls),
    sprintf("(%.1f%%)\n", 100 * mean(y_cls)))
cat("distinct scaffolds:", uniqueN(scaffold), "| scaffolds with a single compound:",
    sum(table(scaffold) == 1), "| largest scaffold family:", max(table(scaffold)), "compounds\n")



The 2,764 compounds carry 914 actives at the 100 nM gate — a 33% positive rate, which is comfortable by the standards of screening data and will be deliberately broken in Section 5.3. The 1,218 scaffolds, 858 of them singletons against a largest family of 207 compounds, are the structure that the rest of this unit exploits: a dataset of a few heavily explored series plus a long tail of one-off molecules.

## 1. Validation answers a question, and the question is about deployment

Every validation protocol is a simulation of the future. Writing it down forces you to say what the future is:

| claim you want to make | future being simulated | split that simulates it |
|---|---|---|
| "this model predicts the potency of further analogues of our series" | more compounds from the same chemotypes | random split |
| "this model finds potent compounds in chemotypes we have not worked on" | a new series, a new vendor catalogue | scaffold or cluster split |
| "this model would have predicted next year's results from last year's data" | the laboratory's own future | time split |
| "this model transfers to another site/cohort" | another laboratory, another hospital | external validation |

The numbers these protocols produce are *different numbers about different questions*, not better and worse estimates of one quantity. The error of the field is to compute the easiest one and write the sentence that belongs to the hardest. Wallach and Heifets showed that most ligand-based classification benchmarks reward memorisation of series rather than generalisation<sup class="c"><a href="#r3">[3]</a></sup>; Sheridan showed on industrial datasets that time-split validation is systematically more pessimistic — and more honest — than random cross-validation<sup class="c"><a href="#r2">[2]</a></sup>; the same complaint runs through the clinical-prediction literature, where TRIPOD+AI now requires the validation design to be stated explicitly<sup class="c"><a href="#r8">[8]</a></sup>.

<div class="box bridge"><h4>Domain context — series, scaffolds and chemotypes</h4>
<p>Medicinal chemists do not make molecules one at a time. They make a <strong>series</strong>: one core structure decorated with dozens of variations, so that a project's data contain large families of near-identical compounds. The computable proxy for "the core" is the <strong>Bemis–Murcko scaffold</strong><sup class="c"><a href="#r11">[11]</a></sup> — the molecule stripped of its side chains, leaving ring systems and the linkers between them. A <strong>chemotype</strong> is a looser notion: a family of structures a chemist would recognise as the same idea. Our 2,764 compounds, curated from the public ChEMBL database<sup class="c"><a href="#r32">[32]</a></sup>, contain 1,218 distinct scaffolds, of which 858 occur once and the largest covers 207 compounds. A random split therefore puts near-twins of most test compounds into the training set, which is exactly what makes it optimistic — and exactly what a <em>scaffold split</em>, which keeps a whole family on one side of the split, is designed to prevent. The clinical analogue is a dataset with several tissue sites per patient, where the family is the patient; the equivalent of the scaffold split is splitting by <code>patient_id</code>.</p></div>

## 2. The split taxonomy, in code

### 2.1 Random and stratified splits

A random split assigns rows to folds independently. A **stratified** split does the same while preserving the proportion of each class in every fold, which matters when the positive class is small: with 5 folds and 90 positives, a random split can produce a fold with 12 positives and one with 24, and the fold-to-fold variance of any metric inflates accordingly. In R, `caret::createFolds()` stratifies automatically when it is given a factor (and bins a numeric target into quantiles), so the unstratified version has to be written explicitly — which is a good way to see what stratification does.



In [ ]:
## Class balance per fold: plain random folds vs stratified folds.
set.seed(0); f_rand  <- split(sample(seq_along(y)), rep(1:5, length.out = length(y)))
set.seed(0); f_strat <- caret::createFolds(factor(y_cls), k = 5)
fb <- rbindlist(list(
  data.table(scheme = "random",     fold = 1:5, active_rate = sapply(f_rand,  function(i) mean(y_cls[i]))),
  data.table(scheme = "stratified", fold = 1:5, active_rate = sapply(f_strat, function(i) mean(y_cls[i])))))
print(ggplot(fb, aes(fold, active_rate, colour = scheme)) +
  geom_hline(yintercept = mean(y_cls), linetype = "dashed", linewidth = 0.4) +
  geom_line() + geom_point(size = 1.6) +
  scale_colour_manual(values = c(random = unname(PAL["orange"]), stratified = unname(PAL["blue"])), name = NULL) +
  labs(x = "fold", y = "fraction of actives in the test fold",
       title = "Stratification removes an avoidable source of variance") +
  theme(legend.position = "top"))
print(fb[, .(mean = round(mean(active_rate), 4), sd = round(sd(active_rate), 4)), by = scheme])
cat("overall active rate:", round(mean(y_cls), 4), "| fold sizes:", sapply(f_rand, length), "\n")



<p class="caption"><b>Figure 1.</b> Fraction of actives in each test fold under plain and stratified 5-fold cross-validation. Both average to the overall rate of 0.331, but the plain folds wander between 0.311 and 0.341 (standard deviation 0.0122) while the stratified folds are flat at 0.330–0.331 (standard deviation 0.0005, a 25-fold reduction). With 914 actives the effect is cosmetic; with 50 actives it is the difference between a usable and an unusable estimate. Stratification costs nothing, so it is the default for classification — and note that it stratifies on the <em>label</em>, which means it cannot be combined naively with a group split (one has to interleave the two constraints by hand, or accept approximate stratification within groups).</p>

### 2.2 Group splits: keeping a family together

A **group split** guarantees that all rows sharing a group label land in the same fold. `caret::groupKFold(groups, k)` takes the group vector — scaffold for molecules, patient identifier for tissue samples, trajectory for simulation frames — and returns folds that never split a group. (It returns *training* index sets rather than test sets, which is the opposite convention from `createFolds`; forgetting that silently inverts the experiment.) This is the mechanism behind both the scaffold split and the cluster split; the only difference between them is how the group label is defined.

### 2.3 The Bemis–Murcko scaffold split

The convention in cheminformatics is deterministic: sort scaffold families by size, largest first, and fill the training set until it reaches the target fraction; everything after that goes to the test set. Large, well-populated series therefore go to training and the rarest, most isolated structures to test — a deliberately pessimistic design that answers "can this model say anything about a chemotype it has barely seen?"



In [ ]:
## A deterministic Bemis-Murcko scaffold split: largest families to training, the tail to test.
scaffold_split <- function(sc, test_size = 0.2) {
  fams <- split(seq_along(sc), sc)
  fams <- fams[order(-lengths(fams), sapply(fams, `[`, 1))]   ## size, then first index: deterministic
  n_train <- round((1 - test_size) * length(sc))
  tr <- integer(0); te <- integer(0)
  for (g in fams) if (length(tr) < n_train) tr <- c(tr, g) else te <- c(te, g)
  list(train = tr, test = te)
}
ss <- scaffold_split(scaffold, 0.2); tr_scaf <- ss$train; te_scaf <- ss$test
cat("scaffold split: train", length(tr_scaf), "test", length(te_scaf), "compounds\n")
cat("scaffolds shared between the two sides:", length(intersect(scaffold[tr_scaf], scaffold[te_scaf])), "\n")
cat(sprintf("active rate  train %.3f | test %.3f   (overall %.3f)\n",
            mean(y_cls[tr_scaf]), mean(y_cls[te_scaf]), mean(y_cls)))
cat(sprintf("mean compounds per scaffold  train %.2f | test %.2f\n",
            length(tr_scaf) / uniqueN(scaffold[tr_scaf]), length(te_scaf) / uniqueN(scaffold[te_scaf])))



The split is clean — no scaffold appears on both sides — and it is *not* balanced: the test side has an active rate of 0.210 against 0.361 in training, because potent compounds concentrate in the large optimised series that went to training. The asymmetry is sharper still in the last line: a training scaffold carries 3.32 compounds on average, a test scaffold exactly 1.00 — the test set is made of singletons by construction. That imbalance is itself information about what the split is simulating, and it is a reason to report PR-AUC alongside ROC-AUC later in this unit.

### 2.4 The cluster split

Scaffolds are a crude definition of "family": two compounds with different ring systems can still be obvious analogues, and a scaffold with a single compound does not mean the compound is isolated. The alternative is to cluster the compounds by fingerprint similarity and use the cluster as the group — in cheminformatics the standard algorithm is Butina's<sup class="c"><a href="#r12">[12]</a></sup>, a single-pass neighbour-count clustering at a fixed Tanimoto cut-off, and the resulting *leave-cluster-out* protocol is the one recommended when the question is prospective performance<sup class="c"><a href="#r13">[13]</a></sup>.

R has no `Butina.ClusterData`, so we write it: compute the full similarity matrix, mark every pair closer than the cut-off, order the molecules by how many neighbours they have, and walk down that list claiming all still-unassigned neighbours of each centroid. That is the whole algorithm.



In [ ]:
## Butina clustering at Tanimoto distance 0.4, implemented from the definition.
tanimoto <- function(A, B) {
  C <- as.matrix(Matrix::tcrossprod(A, B))
  a <- Matrix::rowSums(A); b <- Matrix::rowSums(B)
  C / (outer(a, b, "+") - C)
}
butina <- function(Xb, cutoff = 0.4) {
  nb <- tanimoto(Xb, Xb) >= (1 - cutoff); diag(nb) <- TRUE     ## neighbour mask
  cl <- rep(NA_integer_, nrow(Xb)); k <- 0L
  for (i in order(rowSums(nb), decreasing = TRUE)) {            ## densest centroid first
    if (!is.na(cl[i])) next
    k <- k + 1L; cl[nb[i, ] & is.na(cl)] <- k                   ## claim unassigned neighbours
  }
  cl
}
cluster_id <- butina(X, 0.4)
sz <- as.integer(table(cluster_id))
cat(sprintf("Butina at distance 0.4: %d clusters | largest %d | singletons %d | median size %.0f\n",
            max(cluster_id), max(sz), sum(sz == 1), median(sz)))
cat(sprintf("compounds in the 10 largest clusters: %d (%.0f%% of the set)\n",
            sum(sort(sz, decreasing = TRUE)[1:10]), 100 * sum(sort(sz, decreasing = TRUE)[1:10]) / length(y)))



773 clusters emerge, the largest holding 125 compounds, 465 of them singletons and the ten largest covering 20% of the dataset: a handful of heavily explored series plus a long tail of one-off structures, which is what a public bioactivity table always looks like. (The Python edition, using RDKit's own implementation, reports 767 clusters with the same largest cluster of 125; the handful of clusters that differ come from ties in the neighbour counts, which the two implementations break in different orders.)

### 2.5 A simulated time split

A real time split needs dates. The curated table does not carry them, but ChEMBL molecule identifiers are assigned in roughly chronological order of deposition, so the smallest identifier of a compound is a usable proxy for "when this structure entered the public record". Sorting by it and cutting at 80% simulates training on the past and testing on the future — the protocol Sheridan advocated<sup class="c"><a href="#r2">[2]</a></sup>. This is a *proxy*, and the unit says so: identifier order is not publication date, and a compound re-measured years later keeps its old identifier.



In [ ]:
## Simulated time split: order compounds by their lowest ChEMBL molecule id (a deposition-order proxy).
first_id <- sapply(strsplit(reg$chembl_ids, "\\|"), function(v) min(as.integer(gsub("\\D", "", v))))
ord  <- order(first_id)
cut  <- floor(0.8 * length(ord))
tr_time <- ord[1:cut]; te_time <- ord[(cut + 1):length(ord)]
cat("ChEMBL id range:", min(first_id), "to", max(first_id), "\n")
cat("time split: past", length(tr_time), "compounds, future", length(te_time), "\n")
cat(sprintf("active rate  past %.3f | future %.3f\n", mean(y_cls[tr_time]), mean(y_cls[te_time])))
cat(sprintf("mean pActivity  past %.2f | future %.2f\n", mean(y[tr_time]), mean(y[te_time])))
cat("scaffolds in the future set never seen in the past:",
    length(setdiff(scaffold[te_time], scaffold[tr_time])), "of", uniqueN(scaffold[te_time]), "\n")



The simulated future is measurably different from the simulated past: its active rate is 0.226 against 0.357, its mean potency 6.24 against 6.57, and 210 of its 316 scaffolds are new. This is the drift that a random split assumes away — later compounds in a target's history are the harder, more diverse ones, because the easy chemotypes were explored first.

### 2.6 Distance to the training set: the diagnostic that explains every split

Rather than trusting a split's name, measure what it did. For each test compound, compute the Tanimoto similarity to its nearest training compound. The distribution of that quantity *is* the difficulty of the split, and it is the same quantity that Unit 17 turns into an applicability domain<sup class="c"><a href="#r26">[26]</a></sup>.



In [ ]:
## How far is each test compound from the training set, under three different splits?
set.seed(0)
te_rand <- sort(sample(seq_along(y), round(0.2 * length(y)))); tr_rand <- setdiff(seq_along(y), te_rand)
nn_sim  <- function(tr, te) apply(tanimoto(X[te, ], X[tr, ]), 1, max)
sims <- list(random = nn_sim(tr_rand, te_rand), scaffold = nn_sim(tr_scaf, te_scaf),
             `time (proxy)` = nn_sim(tr_time, te_time))
sd_tab <- rbindlist(lapply(names(sims), function(n) data.table(split = n, sim = sims[[n]])))
print(ggplot(sd_tab, aes(sim, colour = split)) +
  geom_vline(xintercept = 0.5, linetype = "dotted", colour = "grey40") +
  geom_freqpoly(bins = 30, linewidth = 0.7) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green")]), name = NULL) +
  labs(x = "Tanimoto similarity to the nearest training compound", y = "test compounds",
       title = "What each split actually did to the test set") + theme(legend.position = "top"))
for (n in names(sims)) cat(sprintf("%-13s mean %.3f | median %.3f | 10th percentile %.3f | %.0f%% below 0.5\n",
    n, mean(sims[[n]]), median(sims[[n]]), quantile(sims[[n]], 0.1), 100 * mean(sims[[n]] < 0.5)))



<p class="caption"><b>Figure 2.</b> Distance from the test set to the training set under three splits, measured as nearest-neighbour Tanimoto similarity. The random split leaves test compounds a mean similarity of 0.741 from their nearest training neighbour, with only 9% below the 0.5 line that is commonly used as a rough applicability boundary. The scaffold split pushes the mean down to 0.616 and 27% of test compounds below 0.5, with a 10th percentile of 0.304 — it genuinely isolates the test set. The simulated time split sits at 0.730, barely different from random on this diagnostic (only 4% below 0.5) even though, as the next section shows, it is one of the <em>harder</em> splits by performance: later compounds resemble earlier ones structurally but behave differently, which fingerprint similarity cannot see. The diagnostic and the score together say more than either alone.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Build a <em>maximum-dissimilarity</em> split: choose as the test set the 20% of compounds with the lowest mean similarity to the rest of the dataset (one call to <code>tanimoto(X, X)</code> and a <code>rowMeans</code>). Add its nearest-neighbour similarity distribution to Figure 2 and predict, before running anything, whether its ROC-AUC will be above or below the scaffold split's. Then measure it.</p></div>

## 3. The canonical experiment: one model, six protocols

Now the experiment that the rest of this block depends on — the same comparison that benchmark studies of fingerprints<sup class="c"><a href="#r28">[28]</a></sup> and of structure-based screening sets<sup class="c"><a href="#r27">[27]</a></sup> report when they control for dataset bias. The model is fixed — a random forest of 200 trees on the 2,048-bit fingerprints, no tuning — and only the validation protocol changes. Two metrics are reported: ROC-AUC, the probability that a random active is ranked above a random inactive, and average precision (PR-AUC), the area under the precision–recall curve, which is sensitive to how many actives there are. `pROC` computes the first; the second is four lines, written here because seeing them is worth more than importing them.



In [ ]:
## The same model under six validation protocols. (~15 s)
auc_roc <- function(obs, p) as.numeric(pROC::auc(pROC::roc(obs, p, quiet = TRUE,
                                                           levels = c(0, 1), direction = "<")))
auc_pr <- function(obs, p) {                       ## average precision, from the definition
  o <- order(p, decreasing = TRUE); yt <- obs[o]
  tp <- cumsum(yt); precision <- tp / seq_along(yt); recall <- tp / sum(yt)
  sum(diff(c(0, recall)) * precision)
}
rf_prob <- function(tr, te, ntree = 200) {
  m <- ranger(x = Xm[tr, ], y = factor(y_cls[tr]), probability = TRUE,
              num.trees = ntree, seed = 1, num.threads = 4)
  predict(m, Xm[te, ])$predictions[, "1"]
}
holdout <- function(tr, te) { p <- rf_prob(tr, te)
  c(ROC_AUC = auc_roc(y_cls[te], p), PR_AUC = auc_pr(y_cls[te], p)) }
rows <- list(`random hold-out (20%)`   = c(holdout(tr_rand, te_rand), ROC_sd = NA, PR_sd = NA),
             `scaffold hold-out (20%)` = c(holdout(tr_scaf, te_scaf), ROC_sd = NA, PR_sd = NA),
             `time hold-out (last 20%)`= c(holdout(tr_time, te_time), ROC_sd = NA, PR_sd = NA))
cvsets <- list(
  `5-fold stratified CV`       = lapply(f_strat, function(te) list(tr = setdiff(seq_along(y), te), te = te)),
  `5-fold grouped by scaffold` = lapply(caret::groupKFold(scaffold,   k = 5), function(tr) list(tr = tr, te = setdiff(seq_along(y), tr))),
  `5-fold grouped by cluster`  = lapply(caret::groupKFold(cluster_id, k = 5), function(tr) list(tr = tr, te = setdiff(seq_along(y), tr))))
for (nm in names(cvsets)) {
  m <- t(sapply(cvsets[[nm]], function(s) holdout(s$tr, s$te)))
  rows[[nm]] <- c(ROC_AUC = mean(m[, 1]), PR_AUC = mean(m[, 2]), ROC_sd = sd(m[, 1]), PR_sd = sd(m[, 2]))
  cat(sprintf("%-27s test-fold sizes: %s\n", nm, paste(sapply(cvsets[[nm]], function(s) length(s$te)), collapse = " ")))
}
split_table <- data.table(protocol = names(rows), do.call(rbind, rows))
knitr::kable(split_table[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "One untuned random forest, six validation protocols.")



<p class="caption"><b>Table (Section 3).</b> One random forest, six protocols. ROC-AUC falls from 0.896 on a random hold-out to 0.879 under a scaffold hold-out and 0.871 under the simulated time split; the cross-validated versions tell the same story more sharply, 0.901 for stratified folds against 0.852 for scaffold groups and 0.847 for cluster groups. PR-AUC — the metric that cares about how many actives you actually retrieve — is far more sensitive: 0.835 on the random hold-out against 0.650 on the scaffold hold-out and 0.692 on the time split. Note also the fold-to-fold standard deviations in the last two columns: 0.018 and 0.027 for stratified folds, but 0.012/0.041 for scaffold groups and 0.020/0.081 for cluster groups. The PR-AUC spread doubles to quadruples under grouping: a group fold's score depends on which series it happens to isolate, and the printed test-fold sizes show that grouped folds are not even equal in size — from 462 to 743 compounds for the scaffold groups, against 552-553 for the stratified folds. Reporting the first row and writing "the model identifies ROCK1 inhibitors with AUC 0.90" would be a claim the experiment does not support.</p>



In [ ]:
## The same comparison for the regression target, and both as a figure. (~15 s)
rmse <- function(obs, pred) sqrt(mean((obs - pred)^2))
mae  <- function(obs, pred) mean(abs(obs - pred))
r2   <- function(obs, pred) 1 - sum((obs - pred)^2) / sum((obs - mean(obs))^2)
rf_reg <- function(tr, te, ntree = 150) {
  m <- ranger(x = Xm[tr, ], y = y[tr], num.trees = ntree, seed = 1, num.threads = 4)
  predict(m, Xm[te, ])$predictions
}
rmse_rand <- sapply(f_rand, function(te) rmse(y[te], rf_reg(setdiff(seq_along(y), te), te)))
rmse_scaf <- sapply(caret::groupKFold(scaffold, k = 5),
                    function(tr) { te <- setdiff(seq_along(y), tr); rmse(y[te], rf_reg(tr, te)) })
lab <- gsub(" hold-out.*| CV|5-fold ", "", split_table$protocol)
p1 <- ggplot(melt(split_table[, .(lab, ROC_AUC, PR_AUC)], id.vars = "lab"),
             aes(factor(lab, levels = lab[1:6]), value, fill = variable)) +
  geom_col(position = "dodge", width = 0.7) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  ylim(0, 1) + labs(x = NULL, y = "score", title = "Classification: the protocol decides the number") +
  theme(axis.text.x = element_text(angle = 28, hjust = 1, size = 7), legend.position = "top")
rb <- data.table(protocol = c("random 5-fold", "scaffold groups"),
                 RMSE = c(mean(rmse_rand), mean(rmse_scaf)), sd = c(sd(rmse_rand), sd(rmse_scaf)))
p2 <- ggplot(rb, aes(protocol, RMSE)) +
  geom_col(fill = unname(PAL[c("blue", "orange")]), width = 0.5) +
  geom_errorbar(aes(ymin = RMSE - sd, ymax = RMSE + sd), width = 0.15, linewidth = 0.3) +
  geom_text(aes(label = sprintf("%.3f", RMSE)), vjust = -2.2, size = 3) +
  labs(x = NULL, y = "RMSE (log units)", title = "Regression: same model, same data")
print(p1 | p2)
cat(sprintf("regression RMSE: random %.3f (sd %.3f) | scaffold %.3f (sd %.3f) | penalty %.3f log units\n",
            mean(rmse_rand), sd(rmse_rand), mean(rmse_scaf), sd(rmse_scaf), mean(rmse_scaf) - mean(rmse_rand)))
cat("per-fold RMSE, random:", round(rmse_rand, 3), "| scaffold:", round(rmse_scaf, 3), "\n")



<p class="caption"><b>Figure 3.</b> The canonical result of the course. Left: ROC-AUC (blue) and PR-AUC (orange) for one untuned random forest under six protocols; the ordering is the same for both metrics but the PR-AUC gap is an order of magnitude larger for the hold-outs (0.185 against 0.017) and about twice as large for the cross-validated versions (0.090 against 0.049), because precision depends on the prevalence of actives that each split happens to produce. Right: the regression version — RMSE 0.731 log units under random 5-fold cross-validation against 0.793 under scaffold groups, a penalty of 0.063 log units, with fold-to-fold standard deviations of 0.031 and 0.006. The individual folds make the point even better: the random folds span 0.699 to 0.772 while every scaffold fold lies between 0.782 and 0.799, so the <em>best</em> scaffold fold is worse than the worst random fold, and the worst scaffold fold is 14% worse than the best random one. <strong>Both numbers are correct.</strong> The first describes interpolation inside known series, the second extrapolation to new ones, and a paper must say which one it is quoting.</p>

<div class="box key"><h4>Key idea</h4>
<p>A split is a hypothesis about the test set's relationship to the training set. Report the protocol together with the number, give both the optimistic and the pessimistic figure when you have them, and let the reader see the gap. A referee who sees only one number assumes the easiest protocol was used — usually correctly.</p></div>

## 4. Cross-validation: selection versus assessment

### 4.1 k-fold, repeated and the variance of a single estimate

A single 5-fold cross-validation gives one number per fold and one average. That average is itself a random variable: change the seed and it moves. **Repeated** cross-validation (several complete k-fold runs with different partitions) exposes the size of that movement, which is the only honest basis for comparing two scores<sup class="c"><a href="#r25">[25]</a></sup>. The model here is a ridge regression on the fingerprints, whose penalty we fix once with `cv.glmnet` so that the repeated experiment varies nothing but the partition.



In [ ]:
## How much does the cross-validated estimate move when only the partition changes? (~10 s)
set.seed(0); cvg <- cv.glmnet(Xm, y, alpha = 0, nfolds = 5)
LAM <- cvg$lambda.min
ridge_p <- function(tr, te, lam = LAM) {
  m <- glmnet(Xm[tr, ], y[tr], alpha = 0, lambda = lam)
  as.numeric(predict(m, Xm[te, ]))
}
per_fold <- numeric(0); per_run <- numeric(0)
for (r in 1:4) {
  set.seed(r); fo <- split(sample(seq_along(y)), rep(1:5, length.out = length(y)))
  v <- sapply(fo, function(te) rmse(y[te], ridge_p(setdiff(seq_along(y), te), te)))
  per_fold <- c(per_fold, v); per_run <- c(per_run, mean(v))
}
print(ggplot(data.frame(rmse = per_fold), aes(rmse)) +
  geom_histogram(bins = 12, fill = "#9bbce0", colour = "white") +
  geom_vline(xintercept = mean(per_fold), linewidth = 0.5) +
  geom_vline(xintercept = range(per_run), colour = unname(PAL[c("orange", "green")]),
             linetype = "dashed", linewidth = 0.6) +
  labs(x = "RMSE of one fold (log units)", y = "folds",
       title = "Repeated 5-fold CV: 4 repeats x 5 folds of the same ridge model"))
cat(sprintf("ridge penalty fixed at lambda = %.3f (cv.glmnet lambda.min; lambda.1se = %.3f)\n",
            LAM, cvg$lambda.1se))
cat(sprintf("per-fold RMSE: mean %.3f, sd %.3f, min %.3f, max %.3f\n",
            mean(per_fold), sd(per_fold), min(per_fold), max(per_fold)))
cat("per-run (5-fold average) RMSE:", round(per_run, 3),
    sprintf("| standard error of one 5-fold estimate: %.4f\n", sd(per_run)))



<p class="caption"><b>Figure 4.</b> Twenty fold-level RMSE values from four independent 5-fold partitions of the same ridge model (penalty fixed at \(\lambda\) = 1.343). Individual folds range from 0.762 to 0.842 — a spread of 0.08 log units — while the four complete-run averages lie between 0.799 and 0.804 (dashed lines), giving a standard error of the cross-validated estimate of 0.0024. This is the crucial asymmetry: averaging five folds is stable, but the folds themselves are noisy, so a difference of 0.02 between two models <em>on one fold</em> means nothing, whereas a consistent difference of 0.02 across folds and repeats means something. Section 6 turns this into a test.</p>

### 4.2 Selection is not assessment

Choosing a hyperparameter by cross-validation and then quoting the best cross-validated score as the model's performance uses the same data twice: once to choose, once to judge. The result is optimistic, because the maximum of several noisy estimates is biased upwards — the winner's curse. The remedy is **nested cross-validation**: an inner loop selects, an outer loop assesses, and the outer loop never sees the selection<sup class="c"><a href="#r1">[1]</a></sup>.



In [ ]:
## Nested vs non-nested cross-validation on the real dataset (small grid, large n). (~25 s)
grid  <- c(0.01, 0.1, 1, 10)
set.seed(0); outer_folds <- split(sample(seq_along(y)), rep(1:5, length.out = length(y)))
flat <- sapply(grid, function(l)
  mean(sapply(outer_folds, function(te) rmse(y[te], ridge_p(setdiff(seq_along(y), te), te, l)))))
pick_lambda <- function(idx, seed) {                 ## the inner loop: selection only
  set.seed(seed); fo <- split(sample(idx), rep(1:5, length.out = length(idx)))
  sc <- sapply(grid, function(l) mean(sapply(fo, function(te) rmse(y[te], ridge_p(setdiff(idx, te), te, l)))))
  grid[which.min(sc)]
}
nested <- sapply(seq_along(outer_folds), function(i) {
  te <- outer_folds[[i]]; tr <- setdiff(seq_along(y), te)
  lam <- pick_lambda(tr, 100 + i)                    ## chosen inside the outer training set only
  c(rmse = rmse(y[te], ridge_p(tr, te, lam)), lambda = lam)
})
cat("grid RMSE (non-nested):", round(flat, 4), "for lambda =", grid, "\n")
cat(sprintf("non-nested: best lambda %g, reported RMSE %.4f  <- selection and assessment on the same folds\n",
            grid[which.min(flat)], min(flat)))
cat(sprintf("nested    : outer RMSE %.4f (folds %s)\n", mean(nested["rmse", ]),
            paste(round(nested["rmse", ], 3), collapse = ", ")))
cat("            lambda chosen inside each outer fold:", nested["lambda", ], "\n")
cat(sprintf("optimism of the non-nested protocol here: %.4f RMSE units\n", min(flat) - mean(nested["rmse", ])))



On this dataset the two protocols agree to the fourth decimal (0.7968 both) and the inner loop picks \(\lambda = 1\) in every outer fold: with 2,764 compounds and a four-point grid there is almost nothing to overfit, so nested cross-validation is a formality — the optimism is 0.0000 RMSE units. That is a real and reassuring result, and reporting it is better than asserting that nesting "would not have mattered". But it is the opposite of what happens when the grid is large and the dataset small, which is the normal situation in a thesis project. The next chunk shows that regime where it is unambiguous: pure noise, 60 candidate models, and a selection step with nothing to find<sup class="c"><a href="#r1">[1]</a></sup>.



In [ ]:
## The winner's curse, on data with no signal at all: 120 samples, 60 noise features, 60 candidates.
set.seed(0); Xn <- matrix(rnorm(120 * 60), nrow = 120); yn <- rbinom(120, 1, 0.5)
cv_auc <- function(cols, idx, k = 5, seed = 0) {
  set.seed(seed); fo <- split(sample(idx), rep(1:k, length.out = length(idx)))
  mean(sapply(fo, function(te) {
    tr <- setdiff(idx, te)
    m  <- suppressWarnings(glm(yn[tr] ~ Xn[tr, cols], family = binomial))
    auc_roc(yn[te], as.numeric(cbind(1, Xn[te, cols]) %*% coef(m)))
  }))
}
set.seed(1); cand <- lapply(1:60, function(k) sample(60, 8))   ## each candidate = 8 random features
scores <- sapply(cand, cv_auc, idx = seq_len(120))
set.seed(2); of <- split(sample(seq_len(120)), rep(1:5, length.out = 120))
outer_auc <- sapply(of, function(te) {                          ## the whole selection, inside each fold
  tr <- setdiff(seq_len(120), te)
  best <- cand[[which.max(sapply(cand, cv_auc, idx = tr, k = 4, seed = 7))]]
  m <- suppressWarnings(glm(yn[tr] ~ Xn[tr, best], family = binomial))
  auc_roc(yn[te], as.numeric(cbind(1, Xn[te, best]) %*% coef(m)))
})
print(ggplot(data.frame(auc = scores), aes(auc)) +
  geom_histogram(bins = 16, fill = "#9bbce0", colour = "white") +
  geom_vline(xintercept = max(scores), colour = unname(PAL["orange"]), linewidth = 0.8) +
  geom_vline(xintercept = mean(outer_auc), colour = unname(PAL["blue"]), linewidth = 0.8) +
  geom_vline(xintercept = 0.5, linetype = "dashed", linewidth = 0.4) +
  labs(x = "cross-validated ROC-AUC", y = "candidates",
       title = "Selecting the best of 60 models on pure noise"))
cat(sprintf("median candidate %.3f | worst %.3f | best candidate %.3f | nested (honest) %.3f\n",
            median(scores), min(scores), max(scores), mean(outer_auc)))
cat("nested outer folds:", round(outer_auc, 3), "\n")



<p class="caption"><b>Figure 5.</b> Model selection on data with no signal. Sixty candidate feature subsets are cross-validated on 120 rows of Gaussian noise with a coin-flip target; the candidates scatter around AUC 0.5 as they must (median 0.446, worst 0.258), but the <em>best of sixty</em> reaches 0.590 — the orange line, and the number a careless paper would report. Repeating the whole selection inside each outer fold gives 0.432 (blue line, folds 0.386 to 0.514), correctly indistinguishable from chance. The optimism is 0.16 AUC units, and it grows with the number of candidates: a grid search over 500 configurations is 500 lottery tickets.</p>

<div class="box warn"><h4>Common misconception</h4>
<p>"I used cross-validation, so my result is validated." Cross-validation protects against a bad train/test draw; it does nothing about a split that does not match the question, about preprocessing fitted outside the folds (Unit 11), or about selecting among candidates on the same folds you then report. Each of those needs its own guard: the right grouping, a preprocessing chain fitted inside the fold, and nesting.</p></div>

## 5. Metrics, from their definitions

### 5.1 Regression metrics under three splits

RMSE and MAE live in the units of the target (log units of potency), so they can be compared with the experimental noise floor. \(R^2\) is relative to the variance of the evaluation set. Spearman's \(\rho\) measures only the ordering, which is what a screening campaign consumes. The four together say more than any one.



In [ ]:
## All four regression metrics, under the random, scaffold and time splits, on the same forest.
reg_row <- function(nm, tr, te) {
  p <- rf_reg(tr, te, ntree = 200)
  data.table(protocol = nm, RMSE = rmse(y[te], p), MAE = mae(y[te], p), R2 = r2(y[te], p),
             Spearman = cor(y[te], p, method = "spearman"), `sd(y_test)` = sd(y[te]))
}
metr <- rbindlist(list(reg_row("random hold-out",   tr_rand, te_rand),
                       reg_row("scaffold hold-out", tr_scaf, te_scaf),
                       reg_row("time hold-out",     tr_time, te_time)))
knitr::kable(metr[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "The same forest, three protocols, four metrics.")



<p class="caption"><b>Table (Section 5.1).</b> The same forest, three protocols, four metrics. RMSE rises from 0.738 (random) to 0.823 (scaffold) and 0.786 (time); MAE from 0.575 to 0.636 and 0.576; \(R^2\) falls from 0.585 to 0.384 and 0.493; and Spearman \(\rho\) from 0.774 to 0.658 and 0.722 — the ranking degrades less than the absolute accuracy, which is consistent with what the model is for. The last column is a warning about \(R^2\): its denominator is the variance of the particular test set (1.147, 1.050 and 1.105 here), so \(R^2\) mixes model quality with test-set diversity and cannot be compared across protocols. The comparable quantities are RMSE and MAE in log units, read against the experimental noise floor of roughly 0.5 log units (Unit 11, Section 1.1).</p>

### 5.2 The classification metrics, traced by hand

Take ten held-out compounds, the forest's predicted probability for each, and the default decision threshold 0.5. The confusion matrix counts four kinds of outcome, and every classification metric in common use is an arithmetic function of those four numbers.

| | predicted active | predicted inactive |
|---|---|---|
| **truly active** | TP (true positives) | FN (false negatives) |
| **truly inactive** | FP (false positives) | TN (true negatives) |

With the ten compounds printed by the next chunk the counts are TP = 2, FP = 1, FN = 1, TN = 6, from which, by hand:

\[ \text{precision} = \frac{TP}{TP+FP} = \frac{2}{3} = 0.667, \qquad
   \text{recall} = \frac{TP}{TP+FN} = \frac{2}{3} = 0.667, \qquad
   F_1 = \frac{2 \cdot 0.667 \cdot 0.667}{0.667+0.667} = 0.667 \]

\[ \mathrm{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}
   = \frac{2 \cdot 6 - 1 \cdot 1}{\sqrt{3 \cdot 3 \cdot 7 \cdot 7}} = \frac{11}{21} = 0.524 \]

The Matthews correlation coefficient<sup class="c"><a href="#r18">[18]</a></sup> uses all four cells, which is why it is the recommended single-number summary for imbalanced problems where \(F_1\) ignores the true negatives entirely<sup class="c"><a href="#r6">[6]</a></sup>.



In [ ]:
## Confirming the hand calculation, and the full metric panel on the whole held-out set.
proba <- rf_prob(tr_rand, te_rand)
set.seed(11); ten <- sample(length(te_rand), 10)
p10 <- round(proba[ten], 2); t10 <- y_cls[te_rand][ten]; pred10 <- as.integer(p10 >= 0.5)
cm <- function(pred, obs) c(TP = sum(pred == 1 & obs == 1), FP = sum(pred == 1 & obs == 0),
                            FN = sum(pred == 0 & obs == 1), TN = sum(pred == 0 & obs == 0))
k10 <- cm(pred10, t10)
mcc <- function(k) (k[["TP"]] * k[["TN"]] - k[["FP"]] * k[["FN"]]) /
  sqrt(as.numeric(k[["TP"]] + k[["FP"]]) * (k[["TP"]] + k[["FN"]]) *
       (k[["TN"]] + k[["FP"]]) * (k[["TN"]] + k[["FN"]]))
cat("ten compounds: probabilities", p10, "\n               truth        ", t10, "\n")
cat(sprintf("TP=%d FP=%d FN=%d TN=%d\n", k10[["TP"]], k10[["FP"]], k10[["FN"]], k10[["TN"]]))
cat(sprintf("from the counts: precision %.3f recall %.3f F1 %.3f MCC %.3f\n",
            k10[["TP"]] / (k10[["TP"]] + k10[["FP"]]), k10[["TP"]] / (k10[["TP"]] + k10[["FN"]]),
            2 * k10[["TP"]] / (2 * k10[["TP"]] + k10[["FP"]] + k10[["FN"]]), mcc(k10)))
kk <- cm(as.integer(proba >= 0.5), y_cls[te_rand])
cat(sprintf("\nwhole held-out set (n = %d), threshold 0.5:\n", length(te_rand)))
cat(sprintf("  active   : precision %.3f recall %.3f\n", kk[["TP"]] / (kk[["TP"]] + kk[["FP"]]),
            kk[["TP"]] / (kk[["TP"]] + kk[["FN"]])))
cat(sprintf("  inactive : precision %.3f recall %.3f\n", kk[["TN"]] / (kk[["TN"]] + kk[["FN"]]),
            kk[["TN"]] / (kk[["TN"]] + kk[["FP"]])))
cat(sprintf("  accuracy %.3f | MCC %.3f | ROC-AUC %.3f | PR-AUC %.3f\n",
            (kk[["TP"]] + kk[["TN"]]) / length(te_rand), mcc(kk),
            auc_roc(y_cls[te_rand], proba), auc_pr(y_cls[te_rand], proba)))



The hand calculation and the code agree exactly: 0.667, 0.667, 0.667 and 0.524. On the full held-out set the same threshold gives precision 0.788 and recall 0.681 for the active class, against 0.843 and 0.903 for the inactive class, an accuracy of 0.826 and an MCC of 0.607: the minority class is always the harder one, and the default threshold of 0.5 silently favours the majority. Nothing about 0.5 is principled — Section 5.5 moves it.

### 5.3 ROC and precision–recall under imbalance

ROC-AUC is the probability that a random active scores above a random inactive<sup class="c"><a href="#r17">[17]</a></sup>. It is invariant to prevalence, which is both its strength and its trap: in a screening campaign where 1 compound in 1,000 is active, an AUC of 0.9 can still mean that almost every predicted hit is a false positive. The precision–recall curve answers the question the experimentalist asks — "of the compounds I buy, how many will work?" — and its area collapses when prevalence falls<sup class="c"><a href="#r5">[5]</a></sup><sup class="c"><a href="#r19">[19]</a></sup>. The experiment: keep the same model and the same features, and only change the proportion of actives by sub-sampling them.



In [ ]:
## The same classifier on a 33%-active and a 5%-active version of the same problem.
set.seed(0)
act <- which(y_cls == 1); ina <- which(y_cls == 0)
rare_pool <- c(ina, sample(act, round(0.05 / 0.95 * length(ina))))   ## 5% actives
curves <- list(); summ <- list()
for (nm in c("33% active (full set)", "5% active (sub-sampled)")) {
  pool <- if (startsWith(nm, "33")) seq_along(y) else rare_pool
  set.seed(0); te <- pool[caret::createDataPartition(factor(y_cls[pool]), p = 0.25, list = FALSE)]
  tr <- setdiff(pool, te)
  pp <- rf_prob(tr, te)
  o  <- order(pp, decreasing = TRUE); yt <- y_cls[te][o]; tpc <- cumsum(yt)
  roc1 <- pROC::roc(y_cls[te], pp, quiet = TRUE, levels = c(0, 1), direction = "<")
  curves[[nm]] <- list(roc = data.table(set = nm, fpr = 1 - roc1$specificities, tpr = roc1$sensitivities),
                       pr  = data.table(set = nm, recall = tpc / sum(yt), precision = tpc / seq_along(yt)))
  summ[[nm]] <- data.table(set = nm, n_test = length(te), actives = sum(y_cls[te]),
                           prevalence = mean(y_cls[te]), ROC_AUC = auc_roc(y_cls[te], pp),
                           PR_AUC = auc_pr(y_cls[te], pp), Brier = mean((pp - y_cls[te])^2))
}
sm <- rbindlist(summ)
p1 <- ggplot(rbindlist(lapply(curves, `[[`, "roc")), aes(fpr, tpr, colour = set)) +
  geom_abline(linetype = "dashed", linewidth = 0.3) + geom_line(linewidth = 0.7) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "false positive rate", y = "true positive rate", title = "ROC is almost blind to prevalence") +
  theme(legend.position = "top")
p2 <- ggplot(rbindlist(lapply(curves, `[[`, "pr")), aes(recall, precision, colour = set)) +
  geom_hline(yintercept = sm$prevalence, colour = unname(PAL[c("blue", "orange")]),
             linetype = "dotted", linewidth = 0.4) +
  geom_line(linewidth = 0.7) + ylim(0, 1) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "recall", y = "precision", title = "Precision-recall is not") + theme(legend.position = "top")
print(p1 | p2)
print(sm[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 4) else v)])



<p class="caption"><b>Figure 6.</b> The same forest on a 33%-active and a 5%-active version of the ROCK1 problem, with stratified 25% test sets. ROC-AUC falls modestly, from 0.901 to 0.804, but PR-AUC collapses from 0.812 to 0.253 — while the dotted lines show the no-skill baselines, which are the prevalences themselves (0.331 and 0.051). Relative to its baseline the imbalanced model is still informative (0.253 against 0.051 is a five-fold enrichment), but the absolute precision a chemist would experience is low, and the curve is visibly ragged because only 25 actives remain in the test set. The Brier score also <em>improves</em> (0.120 to 0.043) purely because predicting small probabilities for a rare event is easy: no metric computed at a single prevalence transfers to another prevalence, so the prevalence must be reported with the metric.</p>

### 5.4 Calibration: are the probabilities probabilities?

A model is **calibrated** when, among the cases it assigns probability 0.7, about 70% are truly positive. Ranking metrics are indifferent to calibration — ROC-AUC is unchanged by any monotone transformation of the scores — but every decision made with a probability needs it<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>. The **Brier score** is the mean squared error of the predicted probability<sup class="c"><a href="#r21">[21]</a></sup>, and it decomposes into calibration and resolution. R has no `calibration_curve`, so we bin by quantile and average — four lines.



In [ ]:
## Calibration curves: random forest vs penalised logistic regression on the same split.
set.seed(0); te_c <- sample(seq_along(y), round(0.25 * length(y))); tr_c <- setdiff(seq_along(y), te_c)
set.seed(0); cvl <- cv.glmnet(Xm[tr_c, ], y_cls[tr_c], family = "binomial", alpha = 0, nfolds = 5)
probs <- list(`random forest` = rf_prob(tr_c, te_c),
              `logistic (ridge)` = as.numeric(predict(cvl, Xm[te_c, ], s = "lambda.min", type = "response")))
calib <- rbindlist(lapply(names(probs), function(nm) {
  pp <- probs[[nm]]
  data.table(model = nm, pp = pp, obs = y_cls[te_c],
             bin = cut(pp, quantile(pp, 0:8 / 8), include.lowest = TRUE))[
    , .(predicted = mean(pp), observed = mean(obs), n = .N), by = .(model, bin)]
}))
print(ggplot(calib, aes(predicted, observed, colour = model)) +
  geom_abline(linetype = "dashed", linewidth = 0.4) +
  geom_line() + geom_point(size = 1.5) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]), name = NULL) +
  labs(x = "mean predicted probability in the bin", y = "observed fraction of actives",
       title = "Calibration (8 quantile bins)") + theme(legend.position = "top"))
for (nm in names(probs)) cat(sprintf("%-17s Brier %.4f | ROC-AUC %.3f\n", nm,
    mean((probs[[nm]] - y_cls[te_c])^2), auc_roc(y_cls[te_c], probs[[nm]])))
print(calib[, .(predicted = round(predicted, 2), observed = round(observed, 2), n), by = model])



<p class="caption"><b>Figure 7.</b> Calibration of two models with similar ranking ability. The random forest (orange) tracks the diagonal closely over the whole range — predicted 0.02 against observed 0.00 in the lowest bin, 0.86 against 0.87 in the highest — with a Brier score of 0.130. The ridge-penalised logistic regression (blue) is <em>under</em>-confident, which is what a strong penalty does: its predictions are compressed into the band 0.11–0.65 while the observed frequencies in those bins run from 0.02 to 0.84, so it under-states high risks and over-states low ones, and its Brier score is worse (0.153) despite a ROC-AUC only 0.019 lower. This is the opposite failure from the Python edition, whose unpenalised logistic regression on the same bits was <em>over</em>-confident at both ends; the direction of mis-calibration is a property of the regularisation, not of the algorithm family. If the model's output will be multiplied by a cost, fed into a decision rule or shown to a clinician as a risk, calibration must be checked and, if necessary, repaired by Platt scaling or isotonic regression<sup class="c"><a href="#r20">[20]</a></sup>.</p>

### 5.5 The threshold is a decision, not a property of the model

A classifier outputs a score; a threshold turns it into an action. Choosing 0.5 by default silently assumes that a false positive and a false negative cost the same, which in drug discovery and in medicine they never do. Sweep the threshold and read the trade-off<sup class="c"><a href="#r24">[24]</a></sup>.



In [ ]:
## Precision, recall, F1 and MCC as functions of the decision threshold.
ths <- seq(0.05, 0.95, length.out = 37)
sweep <- rbindlist(lapply(ths, function(t) {
  k <- cm(as.integer(proba >= t), y_cls[te_rand])
  data.table(threshold = t,
             precision = if (k[["TP"]] + k[["FP"]] == 0) 0 else k[["TP"]] / (k[["TP"]] + k[["FP"]]),
             recall = k[["TP"]] / (k[["TP"]] + k[["FN"]]),
             F1 = 2 * k[["TP"]] / (2 * k[["TP"]] + k[["FP"]] + k[["FN"]]), MCC = mcc(k))
}))
best_t <- sweep$threshold[which.max(sweep$MCC)]
print(ggplot(melt(sweep, id.vars = "threshold"), aes(threshold, value, colour = variable)) +
  geom_vline(xintercept = 0.5, linetype = "dotted", colour = "grey40") +
  geom_vline(xintercept = best_t, linetype = "dashed", colour = unname(PAL["purple"]), linewidth = 0.5) +
  geom_line(linewidth = 0.7) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green", "purple")]), name = NULL) +
  labs(x = "decision threshold on the predicted probability", y = "metric",
       title = "Every threshold is a different classifier") + theme(legend.position = "top"))
at50 <- sweep[which.min(abs(threshold - 0.5))]
cat(sprintf("at threshold 0.50: precision %.3f recall %.3f F1 %.3f MCC %.3f\n",
            at50$precision, at50$recall, at50$F1, at50$MCC))
cat(sprintf("MCC is maximised at threshold %.2f (MCC %.3f, precision %.3f, recall %.3f)\n",
            best_t, max(sweep$MCC), sweep$precision[which.max(sweep$MCC)], sweep$recall[which.max(sweep$MCC)]))



<p class="caption"><b>Figure 8.</b> Precision, recall, \(F_1\) and MCC against the decision threshold on the held-out set. Precision rises and recall falls monotonically, as they must; \(F_1\) and MCC peak somewhere in between — here MCC is maximised at a threshold of 0.35 (0.640) rather than at 0.5 (0.607), so accepting the default costs 0.033 MCC units and, more tellingly, 15 percentage points of recall (0.827 against 0.681) in exchange for 7 points of precision. In a real project the threshold is set by the cost of the two error types (how many compounds can be bought, how many biopsies are acceptable), not by maximising a symmetric metric; the figure's purpose is to show the size of the trade-off you are making when you accept a default.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Suppose a false positive costs one purchased compound (about 100 units of currency) and a false negative costs a missed lead worth 50 purchased compounds. Write the expected cost as a function of the threshold using the confusion matrix at each threshold (the <code>cm()</code> helper above gives you the four counts), plot it, and find the optimal threshold. Compare it with the MCC-optimal 0.35.</p></div>

## 6. Comparing two models without fooling yourself

Two models, one dataset, a small difference in mean AUC. Is it real? The naive answer applies a paired t-test to the per-fold differences. That test is invalid here, and the reason is worth understanding: cross-validation folds are **not independent**, because every pair of training sets overlaps in roughly \((k-2)/(k-1)\) of its rows. The variance of the mean difference is therefore underestimated, the t statistic is inflated, and the p-value is too small — often by an order of magnitude<sup class="c"><a href="#r14">[14]</a></sup><sup class="c"><a href="#r15">[15]</a></sup>.

Nadeau and Bengio derived the correction<sup class="c"><a href="#r4">[4]</a></sup>: for \(k\) resampled differences \(d_i\) with \(n_{\text{test}}\) test rows and \(n_{\text{train}}\) training rows per split,

\[ t_{\text{corrected}} = \frac{\bar{d}}{\sqrt{\hat{\sigma}^2\left(\frac{1}{k} + \frac{n_{\text{test}}}{n_{\text{train}}}\right)}}, \]

where the extra term \(n_{\text{test}}/n_{\text{train}}\) inflates the standard error to account for the overlap. With 5-fold cross-validation that term is 0.25, which roughly doubles the standard error. A bootstrap confidence interval over the fold differences is the non-parametric companion<sup class="c"><a href="#r16">[16]</a></sup>.



In [ ]:
## Random forest vs penalised logistic regression: 15 paired differences, three ways of judging them. (~20 s)
resamples <- unlist(lapply(1:3, function(r) { set.seed(r); caret::createFolds(factor(y_cls), k = 5) }),
                    recursive = FALSE)
cmp <- t(sapply(resamples, function(te) {
  tr <- setdiff(seq_along(y), te)
  m  <- glmnet(Xm[tr, ], y_cls[tr], family = "binomial", alpha = 0, lambda = cvl$lambda.min)
  c(rf = auc_roc(y_cls[te], rf_prob(tr, te, 150)),
    lr = auc_roc(y_cls[te], as.numeric(predict(m, Xm[te, ], type = "response"))))
}))
d <- cmp[, "rf"] - cmp[, "lr"]
k <- length(d); n_te <- length(resamples[[1]]); n_tr <- length(y) - n_te
t_naive <- mean(d) / (sd(d) / sqrt(k))
t_corr  <- mean(d) / sqrt(var(d) * (1 / k + n_te / n_tr))
set.seed(0); boot <- replicate(2000, mean(sample(d, k, replace = TRUE)))
cat(sprintf("forest AUC %.4f (sd %.4f) | logistic AUC %.4f (sd %.4f)\n",
            mean(cmp[, 1]), sd(cmp[, 1]), mean(cmp[, 2]), sd(cmp[, 2])))
cat(sprintf("mean paired difference: %+.4f over %d resamples (positive in %d of them)\n",
            mean(d), k, sum(d > 0)))
cat(sprintf("naive paired t-test     : t = %.2f, p = %.6f\n", t_naive, 2 * pt(-abs(t_naive), k - 1)))
cat(sprintf("Nadeau-Bengio corrected : t = %.2f, p = %.6f  (correction term n_test/n_train = %.3f)\n",
            t_corr, 2 * pt(-abs(t_corr), k - 1), n_te / n_tr))
cat(sprintf("bootstrap 95%% CI of the difference: [%+.4f, %+.4f]\n", quantile(boot, 0.025), quantile(boot, 0.975)))
cat(sprintf("smallest difference this design could call significant: naive %.4f | corrected %.4f AUC units\n",
            qt(0.975, k - 1) * sd(d) / sqrt(k), qt(0.975, k - 1) * sqrt(var(d) * (1 / k + n_te / n_tr))))



The forest wins by 0.0254 AUC on average and is ahead in all fifteen resamples, so here the verdict survives the correction: the naive t of 14.94 falls to 6.85 — the correction more than halves it, exactly as advertised — but the corrected p-value is still far below 0.05. The last line is the number worth remembering: with this design, the naive test would call a difference of 0.0036 AUC units significant, while the corrected test needs 0.0080 — more than twice as much. The Python edition of this unit compared the forest with an *unpenalised* logistic regression and found a difference of only 0.009 AUC, which is exactly in the band between those two thresholds: there the naive test reported p = 0.0008 and the corrected one p = 0.07, the same data flipping from "significant" to "not". Our penalised logistic regression is weaker, the gap is larger, and the correction does not change the conclusion — which is the outcome you hope for and should still report with the corrected test, not the naive one. And note that even here the practical remark applies: 0.025 AUC units is a small difference, and the choice between the two models should also weigh calibration (Section 5.4), cost and interpretability.



In [ ]:
## The paired differences themselves, which are what the tests are about.
cd <- data.table(resample = seq_len(k), rf = cmp[, "rf"], lr = cmp[, "lr"], d = d)
p1 <- ggplot(melt(cd[, .(resample, rf, lr)], id.vars = "resample"), aes(resample, value, colour = variable)) +
  geom_line() + geom_point(size = 1.3) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]),
                      labels = c("random forest", "logistic (ridge)"), name = NULL) +
  labs(x = "resample (5 folds x 3 repeats)", y = "ROC-AUC", title = "Paired scores move together") +
  theme(legend.position = "top")
p2 <- ggplot(cd, aes(resample, d)) +
  annotate("rect", xmin = 0.5, xmax = k + 0.5, ymin = quantile(boot, 0.025), ymax = quantile(boot, 0.975),
           fill = PAL[["green"]], alpha = 0.15) +
  geom_col(fill = PAL[["blue"]], width = 0.7) +
  geom_hline(yintercept = mean(d), colour = PAL[["green"]], linetype = "dashed", linewidth = 0.5) +
  labs(x = "resample", y = "AUC difference (forest - logistic)",
       title = "The quantity the test is about")
print(p1 | p2)
cat(sprintf("differences: largest %+.4f, smallest %+.4f, sd %.4f\n", max(d), min(d), sd(d)))



<p class="caption"><b>Figure 9.</b> Left: the two models' AUC on each of the fifteen resamples. They rise and fall together — the fold is a comparable source of variation to the model (fold-to-fold standard deviations 0.012 and 0.015 against a mean difference of 0.025), which is exactly why a <em>paired</em> analysis is the right one. Right: the fifteen differences, their mean (+0.0254, dashed line) and the bootstrap 95% interval (shaded, [+0.0221, +0.0286]). The interval is narrow and far from zero, but remember what it describes: the spread of <em>these</em> folds, not of a fresh dataset, so it inherits the same dependence problem as the naive t-test and is not a substitute for the correction. Reporting a figure like this instead of two bare numbers is what makes a model comparison reviewable.</p>

## 7. y-randomisation: is the model learning the target at all?

The oldest sanity check in QSAR<sup class="c"><a href="#r7">[7]</a></sup><sup class="c"><a href="#r10">[10]</a></sup>: permute the target, re-run the entire protocol, and repeat many times. The distribution of scores obtained with scrambled targets is the null distribution of your pipeline — it absorbs every optimism the protocol contains, including the residual leakage that the reproducibility literature finds so often<sup class="c"><a href="#r31">[31]</a></sup>, because a leaky protocol scores above zero even on scrambled targets. The real model's score must sit far outside it.



In [ ]:
## 100 scrambles of the target, the same ridge model each time, 3-fold CV for speed. (~70 s)
set.seed(0); fo3 <- split(sample(seq_along(y)), rep(1:3, length.out = length(y)))
cv_r2 <- function(yy) mean(sapply(fo3, function(te) {
  tr <- setdiff(seq_along(yy), te)
  m  <- glmnet(Xm[tr, ], yy[tr], alpha = 0, lambda = LAM)
  r2(yy[te], as.numeric(predict(m, Xm[te, ])))
}))
real_r2 <- cv_r2(y)
null_r2 <- sapply(1:100, function(s) { set.seed(1000 + s); cv_r2(sample(y)) })
print(ggplot(data.frame(r2 = null_r2), aes(r2)) +
  geom_histogram(bins = 20, fill = "#9bbce0", colour = "white") +
  geom_vline(xintercept = 0, linetype = "dashed", linewidth = 0.4) +
  geom_vline(xintercept = real_r2, colour = unname(PAL["orange"]), linewidth = 0.9) +
  xlim(min(null_r2) - 0.05, real_r2 + 0.08) +
  labs(x = "cross-validated R-squared", y = "scrambles",
       title = "y-randomisation: the null distribution of the whole protocol"))
cat(sprintf("real R2 %.3f | null: mean %.4f, sd %.4f, max %.4f\n",
            real_r2, mean(null_r2), sd(null_r2), max(null_r2)))
cat(sprintf("permutation p-value = (1 + #{null >= real}) / 101 = %.3f | standardised distance z = %.1f\n",
            (1 + sum(null_r2 >= real_r2)) / 101, (real_r2 - mean(null_r2)) / sd(null_r2)))



<p class="caption"><b>Figure 10.</b> The null distribution of 100 y-scrambles against the real score. Scrambled targets give a cross-validated \(R^2\) of −0.153 on average (negative, as they must: a model fitted to noise predicts held-out data worse than the mean) with a standard deviation of 0.020 and a maximum of −0.107; the real target gives 0.458, thirty standard deviations away, and the permutation p-value is at its floor of 1/101 ≈ 0.010. The test cannot "prove" a model is good — it only rules out the hypothesis that the apparent performance comes from the protocol rather than from the data. Notice that the null is centred <em>below</em> zero: any pipeline whose scrambled null sits noticeably above zero has a leak, and this is the cheapest way to find one.</p>

<div class="box warn"><h4>What y-randomisation does and does not test</h4>
<p>It tests the protocol end to end, so it catches leakage, bugs in the fold assignment and target/feature misalignment. It does <em>not</em> test whether the split matches your claim: a leak-free, perfectly scrambled-null protocol on a random split still answers the random-split question. Both checks are needed, and they are independent.</p></div>

## 8. Reporting: the paragraph a referee can check

Everything above collapses into a handful of sentences that must appear in the paper. The next chunk assembles them from the objects in memory, in the order a reader needs them: data and unit of prediction, split and why, protocol, metrics with spread, baseline, sanity checks, and the limits of the claim<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r29">[29]</a></sup>.



In [ ]:
## The validation report, written from the run.
st <- function(p, col) split_table[protocol == p][[col]]
lines <- c(
 sprintf("Data: %d ROCK1 inhibitors with at least one exact measurement (ChEMBL snapshot, curated in Unit 6);", length(y)),
 "  unit of prediction = one standardised parent structure; target = median pActivity; classification gate",
 sprintf("  pActivity >= 7.0 (%d actives, %.0f%%).", sum(y_cls), 100 * mean(y_cls)),
 "Features: Morgan fingerprints, radius 2, 2048 bits, generated with RDKit and loaded from a frozen sparse file.",
 sprintf("Splits: random 80/20 (set.seed(0)); Bemis-Murcko scaffold split (%d test compounds, no shared scaffold);", length(te_scaf)),
 sprintf("  Butina cluster groups (%d clusters at Tanimoto distance 0.4); simulated time split by ChEMBL-identifier", max(cluster_id)),
 "  order (proxy for deposition date).",
 "Model: ranger random forest, 200 trees, seed 1, no hyperparameter tuning.",
 sprintf("Results: ROC-AUC %.3f (random hold-out) vs %.3f (scaffold) vs %.3f (time);",
         st("random hold-out (20%)", "ROC_AUC"), st("scaffold hold-out (20%)", "ROC_AUC"), st("time hold-out (last 20%)", "ROC_AUC")),
 sprintf("  PR-AUC %.3f / %.3f / %.3f;", st("random hold-out (20%)", "PR_AUC"),
         st("scaffold hold-out (20%)", "PR_AUC"), st("time hold-out (last 20%)", "PR_AUC")),
 sprintf("  regression RMSE %.3f +- %.3f (random 5-fold) vs %.3f +- %.3f (scaffold groups).",
         mean(rmse_rand), sd(rmse_rand), mean(rmse_scaf), sd(rmse_scaf)),
 sprintf("Model comparison: forest - ridge logistic = %+.4f AUC over %d resamples; Nadeau-Bengio corrected p = %.4f.",
         mean(d), k, 2 * pt(-abs(t_corr), k - 1)),
 sprintf("Sanity: y-randomisation over 100 scrambles gives R2 = %.3f +- %.3f against %.3f for the real target (p = %.3f).",
         mean(null_r2), sd(null_r2), real_r2, (1 + sum(null_r2 >= real_r2)) / 101),
 "Claim limited to: ranking of compounds inside or near the chemotypes represented in the training set;",
 sprintf("  %.0f%% of scaffold-split test compounds lie below Tanimoto 0.5 to the training set.",
         100 * mean(sims$scaffold < 0.5)))
cat(lines, sep = "\n")



Every clause of that paragraph is a decision this unit has made explicit, and a referee can check each one against the document. The two sentences most often missing from published work are the third (which split, and what it simulates) and the last (the boundary of the claim).

<div class="box bridge"><h4>Bridge — where this unit sits</h4>
<p><a href="../U11_Supervised_Workflow/index.html">Unit 11 (R)</a> built the leakage-free preprocessing chain that this unit evaluates; <a href="../U13_Features_Dimensionality/index.html">Unit 13 (R)</a> performs feature selection inside the folds defined here and measures its stability; <a href="../U14_Ensembles_Tuning/index.html">Unit 14 (R)</a> runs hyperparameter searches inside nested cross-validation under a budget. The <a href="../index.html">R-edition index</a> lists the remaining units — Unit 8 is the statistical background to Section 6, Unit 17 turns the nearest-neighbour diagnostic of Section 2.6 into conformal prediction intervals, Unit 19 replaces ROC-AUC by early-recognition metrics for the ranking question — and the Python counterpart of this unit is <a href="../../Python/U12_Validation/index.html">here</a>. The algorithms compared here are taught in the artificial-intelligence course, whose R edition covers ROC curves and model evaluation in <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">S1</a> and forests in <a href="../../../BGM411/R/U4_Trees_Random_Forest/index.html">U4</a>; the cheminformatics course treats the same scaffold-split lesson for classical QSAR in <a href="../../../Cheminformatics/U14_Validation/index.html">its Unit 14</a>, and the chemical-space view of clustering in <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">its Unit 8</a>.</p></div>

## 9. Summary

You should now be able to explain, without notes:

- that a split is a hypothesis about the future, and which split simulates which claim;
- how to implement random, stratified, group, scaffold, Butina-cluster and time splits in R, and how to diagnose any of them with nearest-neighbour similarity;
- why random splits flatter molecular models, and by how much on this dataset;
- the difference between selection and assessment, why nested cross-validation exists, and when its correction is negligible or large;
- the definitions of precision, recall, \(F_1\), MCC, ROC-AUC, PR-AUC, Brier and calibration, and which of them depend on prevalence;
- that the decision threshold is a cost decision, not a model property;
- why paired cross-validation differences need the Nadeau–Bengio correction, and what a bootstrap interval over folds does and does not say;
- what y-randomisation tests, and why a null centred above zero means a leak;
- the clauses of a validation paragraph.

The numbers to remember from this document: ROC-AUC 0.896 under a random hold-out against 0.879 (scaffold) and 0.871 (time); PR-AUC 0.835 against 0.650 and 0.692; cross-validated ROC-AUC 0.901 (stratified) against 0.852 (scaffold groups) and 0.847 (cluster groups); regression RMSE 0.731 against 0.793 log units; a naive paired t of 14.94 that becomes 6.85 after the correction, which raises the smallest detectable difference from 0.0036 to 0.0080 AUC units; and a y-randomisation null at \(R^2\) = −0.153 against a real score of 0.458.

## 10. Self-assessment

<details><summary>1. Why does a random split overestimate prospective performance for molecules?</summary>Because public bioactivity data are organised into series of near-identical analogues. A random split puts close relatives of most test compounds into training, so the test score mostly measures interpolation within known series. Here the nearest-neighbour similarity of the test set is 0.741 on average under a random split and 0.616 under a scaffold split, with 9% against 27% of compounds below Tanimoto 0.5.</details>

<details><summary>2. You must predict potency for a vendor catalogue your laboratory has never worked with. Which protocol estimates your performance?</summary>A scaffold or cluster split, or better an external set from that catalogue. The random-split number answers a different question. Expect the scaffold-split figure (PR-AUC 0.650 here) rather than the random-split one (0.835).</details>

<details><summary>3. What exactly does a group split guarantee, and what does it not?</summary>It guarantees that all rows sharing a group label fall in the same fold. It does not guarantee balanced classes, equal fold sizes (the grouped test folds here range from 449 to 743 compounds), or that groups are chemically distinct — two different scaffolds can still be close analogues, which is why the cluster split and the similarity diagnostic exist.</details>

<details><summary>4. Your nested and non-nested cross-validation agree to four decimals. Did you waste your time?</summary>No — you measured that the selection bias is negligible for this grid and this sample size (0.0000 RMSE units), which is itself a reportable result. The same comparison with 60 candidates and 120 samples differed by 0.16 AUC units, as the noise experiment shows.</details>

<details><summary>5. ROC-AUC 0.80 and PR-AUC 0.25 on the same predictions. Contradiction?</summary>No. That dataset is 5% active, so the no-skill PR baseline is 0.051 and 0.253 is a five-fold enrichment, while ROC-AUC is nearly prevalence-independent. Report the prevalence with any precision-based metric, and prefer PR-AUC when retrieval of a rare class is the point.</details>

<details><summary>6. Two models have almost identical ROC-AUC but different Brier scores. Which do you deploy?</summary>If the output is a ranking, either; if the output is a probability used in a decision, the better-calibrated one — here the forest (Brier 0.130) over the ridge logistic regression (0.153). ROC-AUC is invariant to any monotone transformation of the scores, so it cannot see calibration at all.</details>

<details><summary>7. Why is the naive paired t-test over cross-validation folds anti-conservative?</summary>Because the folds share training data — any two training sets overlap in most of their rows — so the differences are positively correlated and their sample variance underestimates the true variance of the mean. Nadeau and Bengio add <code>n_test/n_train</code> to the 1/k term, which for 5-fold cross-validation roughly doubles the standard error: our t fell from 14.94 to 6.85.</details>

<details><summary>8. Your y-randomisation null is centred at \(R^2\) = +0.15. What happened?</summary>Something in the protocol gives the model information about the target even when the target is scrambled — preprocessing fitted outside the folds, duplicated rows across folds, or a feature derived from the target. Find it before reading any real score. An honest null sits slightly below zero, like ours at −0.153.</details>

<details><summary>9. Why not just report the best of your six protocols?</summary>Because the protocols answer different questions and the best one is the one that answers the easiest. Reporting the easiest number with the hardest claim is the single most common validation error in the literature, and a reader who sees one number assumes the easiest protocol.</details>

<details><summary>10. How would you validate a model that will be used prospectively next year?</summary>Simulate it: train on data up to a cut-off date and test on everything after it, which is the time split (here approximated by identifier order, which moved the active rate from 0.357 to 0.226 and brought 210 unseen scaffolds). If possible, run the real prospective experiment — predict, then measure — which is the only validation nobody can argue with.</details>

<details><summary>11. What does the standard deviation across cross-validation folds tell you?</summary>How much the estimate depends on which rows happened to be held out — 0.031 RMSE units for the random split and 0.006 for the scaffold-grouped folds here — and note that a small fold-to-fold spread does not mean a good score, since those tightly clustered scaffold folds all sit above the random ones. It is a lower bound on the uncertainty of the point estimate, not a confidence interval for the model's performance on new data, which is wider.</details>

<details><summary>12. Your scaffold-split test set has a much lower active rate than the training set. Is the split broken?</summary>No, it is informative: potent compounds accumulate in the large optimised series that a size-ordered scaffold split sends to training (0.361 active in training against 0.210 in test). It does mean that precision-based metrics are not comparable between the two sides, and that the prevalence must be quoted alongside them.</details>

## Glossary

<dl class="gloss">
<dt>Average precision (PR-AUC)</dt><dd>Area under the precision–recall curve; its no-skill baseline is the prevalence of the positive class.</dd>
<dt>Brier score</dt><dd>Mean squared error of predicted probabilities; lower is better, and its value depends on prevalence.</dd>
<dt>Butina clustering</dt><dd>Single-pass fingerprint clustering at a fixed Tanimoto cut-off: densest centroid first, claiming all unassigned neighbours.</dd>
<dt>Calibration</dt><dd>Agreement between predicted probabilities and observed frequencies; assessed with a reliability curve and the Brier score.</dd>
<dt>Cluster / leave-cluster-out split</dt><dd>A group split whose groups are similarity clusters rather than scaffolds.</dd>
<dt>Group split</dt><dd>A split in which all rows sharing a group label (scaffold, patient, trajectory) stay in the same fold; <code>caret::groupKFold()</code> in R.</dd>
<dt>MCC</dt><dd>Matthews correlation coefficient; a correlation between predicted and true labels that uses all four confusion-matrix cells.</dd>
<dt>Nadeau–Bengio correction</dt><dd>An inflation of the standard error of resampled performance differences by the factor \(1/k + n_{\text{test}}/n_{\text{train}}\), correcting for overlapping training sets.</dd>
<dt>Nested cross-validation</dt><dd>An inner loop that selects hyperparameters and an outer loop that assesses the whole selection procedure.</dd>
<dt>Permutation (y-randomisation) test</dt><dd>Re-running the complete protocol on scrambled targets to obtain the null distribution of its score.</dd>
<dt>Prevalence</dt><dd>The fraction of positives in a dataset; sets the baseline of every precision-based metric.</dd>
<dt>ROC-AUC</dt><dd>Probability that a random positive is scored above a random negative; nearly independent of prevalence.</dd>
<dt>Scaffold split</dt><dd>A deterministic group split by Bemis–Murcko scaffold, largest families to training, rare ones to test.</dd>
<dt>Selection vs assessment</dt><dd>Choosing a model configuration versus estimating the performance of the chosen procedure; using the same data for both is optimistic.</dd>
<dt>Stratified split</dt><dd>A split that preserves class proportions in every fold; the default of <code>caret::createFolds()</code> on a factor.</dd>
<dt>Time split</dt><dd>Training on data up to a cut-off and testing on later data, simulating prospective use.</dd>
<dt>Winner's curse</dt><dd>The upward bias of the maximum of several noisy estimates, which makes the best cross-validated candidate look better than it is.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r2">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r3">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <em>J Chem Inf Model</em>. 2018;58(5):916-932. <a href="https://doi.org/10.1021/acs.jcim.7b00403">doi:10.1021/acs.jcim.7b00403</a></li>
<li id="r4">Nadeau C, Bengio Y. Inference for the generalization error. <em>Mach Learn</em>. 2003;52:239-281. <a href="https://doi.org/10.1023/A:1024068626366">doi:10.1023/A:1024068626366</a></li>
<li id="r5">Saito T, Rehmsmeier M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. <em>PLoS ONE</em>. 2015;10(3):e0118432. <a href="https://doi.org/10.1371/journal.pone.0118432">doi:10.1371/journal.pone.0118432</a></li>
<li id="r6">Chicco D, Jurman G. The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. <em>BMC Genomics</em>. 2020;21:6. <a href="https://doi.org/10.1186/s12864-019-6413-7">doi:10.1186/s12864-019-6413-7</a></li>
<li id="r7">Rücker C, Rücker G, Meringer M. y-Randomization and its variants in QSPR/QSAR. <em>J Chem Inf Model</em>. 2007;47(6):2345-2357. <a href="https://doi.org/10.1021/ci700157b">doi:10.1021/ci700157b</a></li>
<li id="r8">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r9">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r10">Golbraikh A, Tropsha A. Beware of q2! <em>J Mol Graph Model</em>. 2002;20(4):269-276. <a href="https://doi.org/10.1016/S1093-3263(01)00123-1">doi:10.1016/S1093-3263(01)00123-1</a></li>
<li id="r11">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r12">Butina D. Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity: a fast and automated way to cluster small and large data sets. <em>J Chem Inf Comput Sci</em>. 1999;39(4):747-750. <a href="https://doi.org/10.1021/ci9803381">doi:10.1021/ci9803381</a></li>
<li id="r13">Kramer C, Gedeck P. Leave-cluster-out cross-validation is appropriate for scoring functions derived from diverse protein data sets. <em>J Chem Inf Model</em>. 2010;50(11):1961-1969. <a href="https://doi.org/10.1021/ci100264e">doi:10.1021/ci100264e</a></li>
<li id="r14">Dietterich TG. Approximate statistical tests for comparing supervised classification learning algorithms. <em>Neural Comput</em>. 1998;10(7):1895-1923. <a href="https://doi.org/10.1162/089976698300017197">doi:10.1162/089976698300017197</a></li>
<li id="r15">Bouckaert RR, Frank E. Evaluating the replicability of significance tests for comparing learning algorithms. In: <em>PAKDD 2004</em>. LNCS 3056. Springer; 2004:3-12. <a href="https://doi.org/10.1007/978-3-540-24775-3_3">doi:10.1007/978-3-540-24775-3_3</a></li>
<li id="r16">Efron B. Bootstrap methods: another look at the jackknife. <em>Ann Stat</em>. 1979;7(1):1-26. <a href="https://doi.org/10.1214/aos/1176344552">doi:10.1214/aos/1176344552</a></li>
<li id="r17">Hanley JA, McNeil BJ. The meaning and use of the area under a receiver operating characteristic (ROC) curve. <em>Radiology</em>. 1982;143(1):29-36. <a href="https://doi.org/10.1148/radiology.143.1.7063747">doi:10.1148/radiology.143.1.7063747</a></li>
<li id="r18">Matthews BW. Comparison of the predicted and observed secondary structure of T4 phage lysozyme. <em>Biochim Biophys Acta</em>. 1975;405(2):442-451. <a href="https://doi.org/10.1016/0005-2795(75)90109-9">doi:10.1016/0005-2795(75)90109-9</a></li>
<li id="r19">Davis J, Goadrich M. The relationship between precision-recall and ROC curves. In: <em>Proc. 23rd Int. Conf. on Machine Learning (ICML '06)</em>. ACM; 2006:233-240. <a href="https://doi.org/10.1145/1143844.1143874">doi:10.1145/1143844.1143874</a></li>
<li id="r20">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. In: <em>Proc. 22nd Int. Conf. on Machine Learning (ICML '05)</em>. ACM; 2005:625-632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r21">Brier GW. Verification of forecasts expressed in terms of probability. <em>Mon Weather Rev</em>. 1950;78(1):1-3. <a href="https://doi.org/10.1175/1520-0493(1950)078%3C0001:VOFEIT%3E2.0.CO;2">doi:10.1175/1520-0493(1950)078&lt;0001:VOFEIT&gt;2.0.CO;2</a></li>
<li id="r22">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r23">Steyerberg EW, Vickers AJ, Cook NR, et al. Assessing the performance of prediction models: a framework for traditional and novel measures. <em>Epidemiology</em>. 2010;21(1):128-138. <a href="https://doi.org/10.1097/EDE.0b013e3181c30fb2">doi:10.1097/EDE.0b013e3181c30fb2</a></li>
<li id="r24">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565-574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r25">Hawkins DM, Basak SC, Mills D. Assessing model fit by cross-validation. <em>J Chem Inf Comput Sci</em>. 2003;43(2):579-586. <a href="https://doi.org/10.1021/ci025626i">doi:10.1021/ci025626i</a></li>
<li id="r26">Tetko IV, Sushko I, Pandey AK, et al. Critical assessment of QSAR models of environmental toxicity against Tetrahymena pyriformis: focusing on applicability domain and overfitting by variable selection. <em>J Chem Inf Model</em>. 2008;48(9):1733-1746. <a href="https://doi.org/10.1021/ci800151m">doi:10.1021/ci800151m</a></li>
<li id="r27">Sieg J, Flachsenberg F, Rarey M. In need of bias control: evaluating chemical data for machine learning in structure-based virtual screening. <em>J Chem Inf Model</em>. 2019;59(3):947-961. <a href="https://doi.org/10.1021/acs.jcim.8b00712">doi:10.1021/acs.jcim.8b00712</a></li>
<li id="r28">Riniker S, Landrum GA. Open-source platform to benchmark fingerprints for ligand-based virtual screening. <em>J Cheminform</em>. 2013;5:26. <a href="https://doi.org/10.1186/1758-2946-5-26">doi:10.1186/1758-2946-5-26</a></li>
<li id="r29">Walsh I, Fishman D, Garcia-Gasulla D, et al. DOME: recommendations for supervised machine learning validation in biology. <em>Nat Methods</em>. 2021;18(10):1122-1127. <a href="https://doi.org/10.1038/s41592-021-01205-4">doi:10.1038/s41592-021-01205-4</a></li>
<li id="r30">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r31">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r32">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r33">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1-26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r34">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r35">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Chapter 7: model assessment and selection, including the right and wrong ways to cross-validate.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapter 5 on resampling, at an introductory level and in R.</li>
<li>Steyerberg EW. <em>Clinical Prediction Models</em>. 2nd ed. Springer; 2019. — The clinical counterpart of this unit: validation, calibration, decision curves.</li>
<li>Kuhn M, Johnson K. <em>Applied Predictive Modeling</em>. Springer; 2013. — The <code>caret</code> author's treatment of resampling and model comparison.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>caret</code> documentation, "Data Splitting" and "Model Training and Tuning": <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>.</li>
<li><code>pROC</code> vignette on comparing ROC curves (DeLong, bootstrap and permutation tests): <a href="https://cran.r-project.org/package=pROC">cran.r-project.org/package=pROC</a>.</li>
<li>Practical Cheminformatics (P. Walters) — posts on scaffold splits, model comparison and "we need better benchmarks": <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
<li>TRIPOD+AI reporting checklist: <a href="https://www.tripod-statement.org/">tripod-statement.org</a>.</li>
</ul>
